# 04 — Segmentation: who actually responds?

Notebook 03 established that the **Men's email** wins *on average*. But averages hide
targeting opportunities: if the entire effect comes from one customer segment, emailing
everyone wastes sends (and goodwill) on people who will never respond.

**Statistical health warning, stated up front:** subgroup analysis is **exploratory**.
Slice the data enough ways and something will look significant by chance. Discipline here:

1. Segments are defined by **pre-treatment** variables only (past behavior, not outcomes).
2. Every segment family gets a **Holm correction**.
3. Findings are treated as **hypotheses for a follow-up confirmatory test**, not as
   shipping decisions by themselves.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from utils import two_proportion_ztest, holm_correction

sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_csv("../data/raw/hillstrom.csv")

# We segment the WINNER (Mens E-Mail) against control. The Womens arm is
# excluded so every comparison below is a clean two-group test.
sub = df[df["segment"].isin(["Mens E-Mail", "No E-Mail"])].copy()
sub["treated"] = (sub["segment"] == "Mens E-Mail").astype(int)

# Human-readable segment variables, all pre-treatment:
sub["recency_bucket"] = pd.cut(sub["recency"], bins=[0, 3, 6, 9, 12],
                               labels=["1-3 mo", "4-6 mo",
                                       "7-9 mo", "10-12 mo"])
sub["past_mens_buyer"] = np.where(sub["mens"] == 1,
                                  "bought mens before", "no mens history")
sub["customer_age"] = np.where(sub["newbie"] == 1,
                               "new customer", "established")
print(f"{len(sub):,} rows (Mens E-Mail + control only)")

## 1. Conversion lift within each segment

For every level of every segment variable we run the same two-proportion z-test as the main
analysis: treated vs control *within that slice*. The Holm correction is applied across
**all segment tests at once** — the strictest, most defensible choice.

In [ ]:
SEGMENT_VARS = ["recency_bucket", "past_mens_buyer", "customer_age",
                "channel", "zip_code"]

rows = []
for var in SEGMENT_VARS:
    for level, grp in sub.groupby(var, observed=True):
        t = grp[grp["treated"] == 1]
        c = grp[grp["treated"] == 0]
        r = two_proportion_ztest(int(t["conversion"].sum()), len(t),
                                 int(c["conversion"].sum()), len(c))
        rows.append({"variable": var, "segment": str(level),
                     "n": len(grp), "treat rate": r.p_treat,
                     "ctrl rate": r.p_ctrl, "abs lift": r.abs_lift,
                     "ci_low": r.ci_low, "ci_high": r.ci_high,
                     "p raw": r.p_value})

segments = pd.DataFrame(rows)
# One correction across ALL 15 subgroup tests - slicing is where false
# positives breed, so this is where the correction earns its keep.
segments["p Holm"], segments["significant"] = holm_correction(
    segments["p raw"])

segments.sort_values("abs lift", ascending=False).style.format(
    {"treat rate": "{:.3%}", "ctrl rate": "{:.3%}", "abs lift": "{:+.3%}",
     "ci_low": "{:+.3%}", "ci_high": "{:+.3%}",
     "p raw": "{:.2e}", "p Holm": "{:.2e}"})

In [ ]:
# Forest plot: every segment's lift with its CI, grouped by variable.
plot_df = segments.iloc[::-1].reset_index(drop=True)  # nicer top-to-bottom order

fig, ax = plt.subplots(figsize=(9, 0.45 * len(plot_df) + 1.5))
palette = dict(zip(SEGMENT_VARS, sns.color_palette("deep", len(SEGMENT_VARS))))

for i, row in plot_df.iterrows():
    ax.errorbar(row["abs lift"] * 100, i,
                xerr=[[(row["abs lift"] - row["ci_low"]) * 100],
                      [(row["ci_high"] - row["abs lift"]) * 100]],
                fmt="o" if row["significant"] else "s",
                capsize=4, markersize=8,
                color=palette[row["variable"]],
                alpha=1.0 if row["significant"] else 0.45)

ax.axvline(0, color="crimson", ls="--", lw=1.2)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["variable"] + " = " + plot_df["segment"])
ax.set_xlabel("conversion lift vs control (percentage points)")
ax.set_title("Mens E-Mail effect by segment "
             "(filled circle = significant after Holm)")
fig.tight_layout()
fig.savefig("../assets/segmentation_forest.png", dpi=150,
            bbox_inches="tight")
plt.show()

## 2. Are the differences BETWEEN segments real? (interaction tests)

A larger lift in one segment than another can itself be a fluke. The formal check is an
**interaction test**: fit a logistic regression of conversion on treatment, the segment
variable, and their product term. If the interaction coefficient is significant, the
treatment effect genuinely *differs* across segments — that's the statistical license to
target.

In [ ]:
# Logistic regression with a treatment x segment interaction, one variable
# at a time. The likelihood-ratio test compares the model with vs without
# the interaction term - the cleanest "is targeting justified?" test.
import scipy.stats as st

for var in ["past_mens_buyer", "recency_bucket", "customer_age"]:
    full = smf.logit(f"conversion ~ treated * C({var})", data=sub).fit(disp=0)
    reduced = smf.logit(f"conversion ~ treated + C({var})", data=sub).fit(disp=0)
    lr_stat = 2 * (full.llf - reduced.llf)
    dof = full.df_model - reduced.df_model
    p = st.chi2.sf(lr_stat, dof)
    print(f"{var:<18} LR = {lr_stat:6.2f}, dof = {dof:.0f}, "
          f"interaction p = {p:.4f}"
          + ("   <-- effect differs across segments" if p < 0.05 else ""))

## 3. What this means for the business

Read the forest plot and interaction tests together (exact numbers are in the table above):

- The lift is **not uniform**: some segments carry substantially more of the effect than
  others, and the interaction tests tell us which of those differences are statistically
  real rather than slicing noise.
- Where the interaction is significant, a **targeted send** beats a blast: same revenue
  from fewer emails, which protects the scarcest asset in email marketing — subscriber
  attention (every irrelevant send raises unsubscribe risk).
- Where the interaction is *not* significant, resist the temptation to build targeting
  rules on the point estimates. That's how teams ship noise.

**The disciplined next step** is not "roll out targeting tomorrow" — it is a follow-up
**confirmatory experiment** that pre-registers the winning segments as its primary
hypothesis. Exploration generates the hypothesis; only a fresh test confirms it. That
two-step loop (explore → confirm) is exactly how mature experimentation teams operate.